# LLM classifier, dev split: Qwen2.5-7B-Instruct, zero-shot

Runs `scripts/llm_classify.py`: an open LLM (Apache 2.0, no account needed) reads each citing
sentence with the top-20 sentences our reranker retrieved, and its probabilities for A (ACCURATE),
B (NOT ACCURATE) and C (IRRELEVANT / NOT ENOUGH INFO) are the prediction. No training.

**Dev split only (316 citations).** Prompts are compared on dev; test is run once, at the end,
with whatever wins here.

**Before running:** Runtime → Change runtime type → **T4 GPU**. Then Runtime → Run all and allow
Google Drive access. About 15 minutes (model download about 5, classification about 10).
Predictions are saved to `MyDrive/CiteAudit_runs/llm` as they are made, so a disconnected run resumes.

For reference on the same dev evidence: released MultiVerS macro-F1 0.674, our current best
(MultiVerS + BiomedBERT ensemble) 0.698.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')
RES = '/content/drive/MyDrive/CiteAudit_runs/llm'
import os; os.makedirs(RES, exist_ok=True)

%cd /content
!rm -rf CiteAudit && git clone -q https://github.com/SapnilPatel/CiteAudit.git
%cd /content/CiteAudit
!git log -1 --oneline
!pip -q install "transformers>=4.44" accelerate bitsandbytes

## Classify dev

In [ ]:
MODEL, PROMPT = 'Qwen/Qwen2.5-7B-Instruct', 'v1'
OUT = f'{RES}/qwen7b_{PROMPT}_dev.jsonl'
!python scripts/llm_classify.py --data-dir data_train/target/ours_20 --split dev \
    --model {MODEL} --prompt {PROMPT} --load-4bit --resume --out {OUT} 2>&1 | grep -v Warning

## Score

In [ ]:
!python scripts/score_multivers.py {OUT} --split dev
import json
rows = [json.loads(l) for l in open(OUT)]
print('mean probability on the answer letters:', round(sum(r['answer_mass'] for r in rows) / len(rows), 4))

## Download

Downloads the dev predictions. Put the file in the repository root and tell Claude.

In [ ]:
from google.colab import files
files.download(OUT)